# Individual Assignment 2 figure notebook

This notebook regenerates the four standalone 300 dpi figures from the unchanged merged OWID–WDI CSV. Run the cells in order with Python 3 and `pandas`, `numpy` and `matplotlib`. The two selected submission figures are the RQ2 peer paths and the RQ3 change scatter; the other two are supplementary.

The 2000–2021 comparisons require every measure used by that question to be present at **both endpoints**. Missing values are not filled. RQ2 has 190 complete economy pairs; RQ3 has 182. Figures are saved under `working/figures/` and paired data under `working/derived/`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Run from the notebook folder, assignment folder, or University workspace.
cwd = Path.cwd().resolve()
candidates = (cwd, *cwd.parents,
              cwd / "2026-WS/information-visualization/assignments/individual-assignment-02",
              cwd / "assignments/individual-assignment-02")
ROOT = next((p for p in candidates
             if (p / "instructions/original-assignment-brief.docx").exists()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from individual-assignment-02, its subfolders, or University")
DATA = ROOT.parent / "group-assignment-01/working/data/merged/co2_wdi_merged.csv"
FIG = ROOT / "working/figures"
DERIVED = ROOT / "working/derived"
FIG.mkdir(parents=True, exist_ok=True)
DERIVED.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
assert df.shape == (10874, 26) and df.iso3.nunique() == 202
assert not df.duplicated(["iso3", "year"]).any()
assert (df.year.min(), df.year.max()) == (1960, 2024)
R2 = ["co2_pc_wdi_t", "gdp_pc_ppp_int2021", "urban_population_pct"]
R3 = ["renewable_final_energy_pct", "carbon_intensity_energy_g_kwh",
      "energy_pc_owid_kwh", "gdp_pc_ppp_int2021"]

def pair_changes(columns):
    x = df[df.year.isin([2000, 2021])].dropna(subset=columns)
    x = x[x.groupby("iso3").iso3.transform("size") == 2].set_index(["iso3", "year"])
    a, b = x.xs(2000, level="year"), x.xs(2021, level="year")
    out = a[["country", "region_wdi", "population_wdi"]].copy()
    for col in columns:
        out[col + "_2000"] = a[col]
        out[col + "_2021"] = b[col]
        out["change_" + col] = b[col] - a[col]
        out["pct_change_" + col] = np.where(a[col] != 0, 100 * (b[col] / a[col] - 1), np.nan)
    return out.reset_index()

r2, r3 = pair_changes(R2), pair_changes(R3)
assert len(r2) == 190 and len(r3) == 182
r2.to_csv(DERIVED / "rq2_2000_2021_complete_pairs.csv", index=False)
r3.to_csv(DERIVED / "rq3_2000_2021_complete_pairs.csv", index=False)

plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":10,
    "axes.spines.top":False, "axes.spines.right":False,
    "axes.labelcolor":"#28333c", "text.color":"#17232b",
    "axes.edgecolor":"#87949d", "xtick.color":"#42515b", "ytick.color":"#42515b",
    "figure.facecolor":"white", "axes.facecolor":"white", "savefig.facecolor":"white"})
navy, teal, orange = "#24558a", "#008b83", "#d27723"

def save(fig, filename):
    fig.savefig(FIG / filename, dpi=300, bbox_inches="tight", pad_inches=.16)
    plt.close(fig)


## RQ2 selected figure  Peer paths

Estonia and Poland, then Romania and Türkiye, provide two illustrative pairs with fairly similar income and urban shares in 2000 but different later CO₂ paths. Each annual measure is indexed to its **own 2000 value = 100**. All selected series are checked for missing years.


In [ ]:
# Figure 1: the two peer pairs were chosen from baseline GDP/urban proximity,
# with different observed emissions paths. No selected series is interpolated.
selected = ["EST", "POL", "ROU", "TUR"]
fig, axes = plt.subplots(2, 2, figsize=(11.8, 7.3), sharex=True, sharey=True)
fig.suptitle("Similar starting conditions, different emissions paths", fontsize=17, weight="bold", y=.995)
fig.text(.5, .945, "Four economies | all measures indexed to 2000 = 100 | observed years through 2021", ha="center", fontsize=10)
styles = [("co2_pc_wdi_t", "CO₂ per person", navy),
          ("gdp_pc_ppp_int2021", "PPP GDP per person", teal),
          ("urban_population_pct", "Urban share", orange)]
for ax, code in zip(axes.flat, selected):
    z = df[(df.iso3 == code) & df.year.between(2000, 2021)].set_index("year").reindex(range(2000, 2022))
    base = z.loc[2000]
    assert pd.notna(base[R2]).all() and pd.notna(z.loc[2021, R2]).all()
    assert z[R2].notna().all().all(), f"Unexpected gap in {code}"
    for col, lab, color in styles:
        yy = 100 * z[col] / base[col]
        ax.plot(yy.index, yy.values, color=color, lw=2.3, label=lab)
        ax.scatter([2021], [yy.loc[2021]], color=color, s=20, zorder=3)
    name = z.country.dropna().iloc[0]
    ax.set_title(f"{name}  |  2000: ${base.gdp_pc_ppp_int2021:,.0f}, {base.urban_population_pct:.0f}% urban", loc="left", fontsize=10, weight="bold")
    ax.set_ylim(55, 275); ax.set_xlim(2000, 2021.7)
    ax.set_xticks([2000, 2005, 2010, 2015, 2021])
    ax.grid(axis="y", color="#dce3e8", lw=.7)
for ax in axes[:,0]: ax.set_ylabel("Index (2000 = 100)")
for ax in axes[-1,:]: ax.set_xlabel("Year")
fig.legend(*axes.flat[0].get_legend_handles_labels(), loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(.5, -.01))
fig.text(.5, -.06, "GDP is in constant 2021 international dollars; urban share uses national definitions.", ha="center", fontsize=8.4)
fig.tight_layout(rect=[0,.035,1,.92], h_pad=2.2, w_pad=1.4)
save(fig, "rq2_peer_paths.png")


![RQ2 peer paths](../figures/rq2_peer_paths.png)


## RQ2 supplementary figure  All-economy endpoint changes

Each point represents one of the 190 economies with complete 2000 and 2021 values. Horizontal and vertical positions are percentage changes; colour is the urban-share change in percentage points.


In [ ]:
# Figure 2: global context for the selected peer examples.
fig, ax = plt.subplots(figsize=(11, 7))
x = r2.pct_change_gdp_pc_ppp_int2021
y = r2.pct_change_co2_pc_wdi_t
c = r2.change_urban_population_pct
sc = ax.scatter(x, y, c=c, cmap="PuOr_r", vmin=-20, vmax=40, s=37,
                edgecolor="white", linewidth=.35, alpha=.83)
ax.axhline(0, color="#65727c", lw=.9); ax.axvline(0, color="#65727c", lw=.9)
ax.set_yscale("symlog", linthresh=60)
ax.set_yticks([-70, -40, 0, 40, 100, 300, 1000])
ax.yaxis.set_major_formatter(FuncFormatter(lambda v,p:f"{v:g}"))
ax.set_xlim(-55, 430); ax.set_ylim(-80, 1900)
ax.set_xlabel("Change in PPP GDP per person, 2000–2021 (%)")
ax.set_ylabel("Change in CO₂ per person, 2000–2021 (%)")
ax.grid(color="#e3e8eb", lw=.6); ax.set_axisbelow(True)
for code in selected:
    q = r2[r2.iso3 == code].iloc[0]
    ax.annotate(code, (q.pct_change_gdp_pc_ppp_int2021, q.pct_change_co2_pc_wdi_t),
                xytext=(5,5), textcoords="offset points", fontsize=8, weight="bold")
cb = fig.colorbar(sc, ax=ax, pad=.02, shrink=.85)
cb.set_label("Change in urban share (percentage points)")
fig.suptitle("Income growth often coincided with higher emissions per person", x=.12, ha="left", fontsize=17, weight="bold")
fig.text(.12,.925,"2000–2021 | 190 economies with all three measures at both endpoints | labelled peer examples", fontsize=10)
fig.text(.12,.015,"Vertical scale is linear near zero and compressed beyond ±60%; CO₂ percentage changes can be large from small baselines.", fontsize=8.2)
fig.tight_layout(rect=[.08,.035,1,.89])
save(fig,"rq2_all_economies_changes.png")


![RQ2 all-economy changes](../figures/rq2_all_economies_changes.png)


## RQ3 baseline groups

The RQ3 figures use the 182 economies with renewable share, carbon intensity, primary energy use per person and PPP GDP per person at both endpoints. Income and energy-use thirds are calculated from **2000 values within this sample**; they are not World Bank income classes. Both plotted changes equal **2021 value minus 2000 value**.


In [ ]:
# Baseline income and primary-energy tertiles are defined on the exact 182-economy panel.
r3["income_group"] = pd.qcut(r3.gdp_pc_ppp_int2021_2000, 3, labels=["Lower", "Middle", "Higher"])
r3["energy_group"] = pd.qcut(r3.energy_pc_owid_kwh_2000, 3, labels=["Low", "Medium", "High"])
r3.to_csv(DERIVED / "rq3_2000_2021_grouped.csv", index=False)
income_cuts = r3.gdp_pc_ppp_int2021_2000.quantile([1/3, 2/3]).to_numpy()
energy_cuts = r3.energy_pc_owid_kwh_2000.quantile([1/3, 2/3]).to_numpy()


## RQ3 selected figure  Endpoint-change scatter

Each dot is an economy, not a year. The x-axis is change in renewable share of **final** energy; the y-axis is change in CO₂ per unit of **primary** energy. Income panels and energy-use colours use 2000 baselines. The original year-by-year paths are not shown.


In [ ]:
# Figure 3: facet by baseline income, encode baseline energy use by colour.
fig, axes = plt.subplots(1, 3, figsize=(10.6, 6.0), sharey=True)
energy_colors = {"Low":"#487bb1", "Medium":"#d18a25", "High":"#7a4f8c"}
energy_labels = {
    "Low":f"Low: ≤{energy_cuts[0]/1000:.1f}k",
    "Medium":f"Medium: {energy_cuts[0]/1000:.1f}–{energy_cuts[1]/1000:.1f}k",
    "High":f"High: >{energy_cuts[1]/1000:.1f}k",
}
income_labels = {
    "Lower":f"≤ ${income_cuts[0]/1000:.1f}k/person",
    "Middle":f"${income_cuts[0]/1000:.1f}–{income_cuts[1]/1000:.1f}k/person",
    "Higher":f"> ${income_cuts[1]/1000:.1f}k/person",
}
for ax, income in zip(axes, ["Lower", "Middle", "Higher"]):
    z = r3[r3.income_group == income]
    for en in ["Low", "Medium", "High"]:
        q = z[z.energy_group == en]
        ax.scatter(q.change_renewable_final_energy_pct, q.change_carbon_intensity_energy_g_kwh,
                   s=39, c=energy_colors[en], alpha=.8, edgecolors="white", linewidths=.35, label=energy_labels[en])
    ax.axhline(0, color="#64727d", lw=.9); ax.axvline(0,color="#64727d",lw=.9)
    ax.set_title(f"{income} 2000 income | n = {len(z)}\n{income_labels[income]}", fontsize=11.2, weight="bold")
    ax.set_xlim(-45,34); ax.set_xticks([-40,-20,0,20])
    ax.set_yscale("symlog", linthresh=50)
    ax.set_ylim(-1250,210); ax.set_yticks([-1000,-300,-100,-50,0,50,150])
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,p:f"{v:g}"))
    ax.grid(color="#e5eaed",lw=.6); ax.set_axisbelow(True)
    ax.set_xlabel("Renewable share change (pp)", fontsize=11.2)
    ax.tick_params(labelsize=10.5)
axes[0].set_ylabel("CO₂ intensity change (g/kWh primary energy)", fontsize=11.2)
fig.legend(*axes[0].get_legend_handles_labels(), title="Point colour: 2000 primary energy use (kWh/person; k = 1,000)", ncol=3,
           loc="lower center", bbox_to_anchor=(.5,.03), frameon=False, fontsize=10.5, title_fontsize=10.5)
fig.suptitle("Renewable-share and carbon-intensity changes by starting income", x=.04, ha="left", fontsize=15.2, weight="bold")
fig.text(.04,.91,"Each dot: 2021 value minus 2000 value for one of 182 economies; intervening years are not shown.", fontsize=10.4)
fig.text(.04,.015,"pp = percentage points. Income: 2000 PPP GDP/person in constant 2021 international dollars. Y scale compresses beyond ±50 g/kWh.", fontsize=9.1)
fig.tight_layout(rect=[.005,.13,1,.865], w_pad=.8)
save(fig,"rq3_changes_by_income.png")


![RQ3 endpoint-change scatter](../figures/rq3_changes_by_income.png)


## RQ3 supplementary figure  Group median heatmap

Cell colour is the median 2000–2021 carbon-intensity change among economies in each 2000 income-by-energy-use group. Cell counts are printed because group sizes are uneven.


In [ ]:
# Figure 4: a complementary cross-classification, with one quantity encoded by colour.
cells = r3.groupby(["income_group","energy_group"], observed=True).agg(
    n=("iso3","size"), median_intensity_change=("change_carbon_intensity_energy_g_kwh","median"),
    median_renewable_change=("change_renewable_final_energy_pct","median")).reset_index()
cells.to_csv(DERIVED / "rq3_tertile_medians.csv", index=False)
arr = cells.pivot(index="income_group", columns="energy_group", values="median_intensity_change").loc[["Lower","Middle","Higher"],["Low","Medium","High"]]
fig, ax = plt.subplots(figsize=(8.7,6.5))
im=ax.imshow(arr, cmap="PuOr_r", vmin=-75,vmax=75)
ax.set_xticks(range(3), ["Low","Medium","High"]);ax.set_yticks(range(3),["Lower","Middle","Higher"])
ax.set_xlabel("2000 primary energy use per person (within-sample tertiles)")
ax.set_ylabel("2000 PPP GDP per person (within-sample tertiles)")
for i, income in enumerate(["Lower","Middle","Higher"]):
 for j, energy in enumerate(["Low","Medium","High"]):
    match=cells[(cells.income_group==income)&(cells.energy_group==energy)]
    label = "No economies" if match.empty else f"{match.iloc[0].median_intensity_change:+.0f} g/kWh\nn = {int(match.iloc[0].n)}"
    ax.text(j,i,label,ha="center",va="center",fontsize=11,weight="bold")
cb=fig.colorbar(im,ax=ax,shrink=.8);cb.set_label("Median CO₂ intensity change (g/kWh)")
fig.suptitle("Median intensity changes vary across income and energy-use groups",x=.06,ha="left",fontsize=15,weight="bold")
fig.text(.06,.91,"2000–2021 | 182 economies with all four measures at both endpoints | each cell is a median",fontsize=9.5)
fig.text(.06,.015,"Grouping uses 2000 values; uneven cell sizes are printed. Final-energy renewable-share changes are in the supporting CSV.",fontsize=8.2)
fig.tight_layout(rect=[.04,.04,1,.88])
save(fig,"rq3_income_energy_heatmap.png")


![RQ3 income and energy heatmap](../figures/rq3_income_energy_heatmap.png)
